# Notebook 11 — Question-Level Hallucination Risk Prediction

## Two-Stage Prediction Framework

This notebook implements **Stage 1** of our two-stage hallucination prediction system:

**Stage 1 (This notebook):** Question Risk Assessment
- Input: Question/prompt only (no response)
- Output: Probability that the question will elicit a hallucination
- Use case: Pre-generation risk screening

**Stage 2 (Notebooks 02-03):** Response Verification
- Input: Question + response
- Output: Hallucination detection
- Use case: Post-generation verification

## Research Questions
1. Can we predict hallucination risk from question characteristics alone?
2. Which question features are most predictive?
3. How does question-level prediction compare to response-level detection?
4. Can question-risk scores help prioritize verification resources?

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Repo bootstrap
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.load_splits import load_splits
from src.features.question_features import add_question_features, get_question_feature_cols, FEATURE_HYPOTHESES
from src.utils.experiment import seed_everything, make_run_dir, save_metrics_csv
from src.utils.eval import evaluate_split_with_roc

# Reproducibility
SEED = 42
seed_everything(SEED)

# Output directory
REPORTS_DIR = ROOT / "reports"
RUN_DIR = make_run_dir(REPORTS_DIR, "nb11_question_risk", timestamp=False)
PLOTS_DIR = RUN_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

print(f"ROOT: {ROOT}")
print(f"Output directory: {RUN_DIR.relative_to(ROOT)}")

## Load Data and Extract Question Features

In [ ]:
train_df, val_df, test_df = load_splits(ROOT)

print("Original shapes:")
print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")

# Add question features
train_q = add_question_features(train_df)
val_q   = add_question_features(val_df)
test_q  = add_question_features(test_df)

q_features = get_question_feature_cols(train_q)
print(f"\nExtracted {len(q_features)} question features")
print(f"Feature list: {q_features[:10]}...")

## Exploratory Analysis: Question Features vs Hallucination Labels

In [ ]:
# Compute correlations
correlations = train_q[q_features + ['label']].corr()['label'].drop('label').sort_values(ascending=False)

print("Top 10 features correlated with hallucination:")
print(correlations.head(10))
print("\nTop 10 features anti-correlated with hallucination:")
print(correlations.tail(10))

# Save correlations
correlations.to_csv(RUN_DIR / "question_feature_correlations.csv")

In [ ]:
# Plot top correlations
top_n = 15
top_features = correlations.abs().nlargest(top_n)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['red' if x < 0 else 'green' for x in correlations[top_features.index]]
ax.barh(range(len(top_features)), correlations[top_features.index], color=colors, alpha=0.7)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features.index)
ax.set_xlabel('Correlation with Hallucination Label')
ax.set_title(f'Top {top_n} Question Features (by absolute correlation)')
ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "question_feature_correlations.png", dpi=150, bbox_inches='tight')
plt.show()

## Build Question-Risk Prediction Model

We train a classifier using **question features only** to predict hallucination risk.

Labels come from actual responses, but the model has no access to response text.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, roc_curve

# Prepare data
X_train = train_q[q_features]
y_train = train_q["label"].astype(int)

X_val = val_q[q_features]
y_val = val_q["label"].astype(int)

X_test = test_q[q_features]
y_test = test_q["label"].astype(int)

print(f"Training on {len(q_features)} question features")
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Val:   X={X_val.shape}, y={y_val.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")

In [ ]:
# Train Logistic Regression (interpretable baseline)
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED))
])

lr_pipeline.fit(X_train, y_train)

# Evaluate
lr_train_metrics = evaluate_split_with_roc("Train (Question-LR)", lr_pipeline, X_train, y_train, verbose=True)
lr_val_metrics   = evaluate_split_with_roc("Val (Question-LR)", lr_pipeline, X_val, y_val, verbose=True)
lr_test_metrics  = evaluate_split_with_roc("Test (Question-LR)", lr_pipeline, X_test, y_test, verbose=True)

In [ ]:
# Train Random Forest (non-linear baseline)
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=SEED, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)

# Evaluate
rf_train_metrics = evaluate_split_with_roc("Train (Question-RF)", rf_pipeline, X_train, y_train, verbose=True)
rf_val_metrics   = evaluate_split_with_roc("Val (Question-RF)", rf_pipeline, X_val, y_val, verbose=True)
rf_test_metrics  = evaluate_split_with_roc("Test (Question-RF)", rf_pipeline, X_test, y_test, verbose=True)

## Compare Question-Risk vs Response-Detection

Load Stage 2 (response-based detection) results for comparison.

In [ ]:
# Load response-based detection results (from NB03)
nb03_metrics = pd.read_csv(ROOT / "reports" / "nb03_feature_based" / "metrics.csv")
response_test_f1 = nb03_metrics[nb03_metrics['split'] == 'Test']['f1'].values[0]
response_test_auc = nb03_metrics[nb03_metrics['split'] == 'Test']['roc_auc'].values[0]

# Comparison table
comparison = pd.DataFrame([
    {
        "Stage": "1: Question Risk (LR)",
        "Input": "Question only",
        "Test F1": lr_test_metrics.f1,
        "Test ROC-AUC": lr_test_metrics.roc_auc,
        "Test Accuracy": lr_test_metrics.accuracy,
    },
    {
        "Stage": "1: Question Risk (RF)",
        "Input": "Question only",
        "Test F1": rf_test_metrics.f1,
        "Test ROC-AUC": rf_test_metrics.roc_auc,
        "Test Accuracy": rf_test_metrics.accuracy,
    },
    {
        "Stage": "2: Response Verification",
        "Input": "Question + Response",
        "Test F1": response_test_f1,
        "Test ROC-AUC": response_test_auc,
        "Test Accuracy": nb03_metrics[nb03_metrics['split'] == 'Test']['accuracy'].values[0],
    }
])

print("\n" + "="*80)
print("TWO-STAGE FRAMEWORK PERFORMANCE COMPARISON")
print("="*80)
print(comparison.to_string(index=False))
print("="*80)

comparison.to_csv(RUN_DIR / "two_stage_comparison.csv", index=False)

## Feature Importance Analysis

In [ ]:
# Logistic Regression coefficients
lr_clf = lr_pipeline.named_steps['clf']
coefs = pd.DataFrame({
    'feature': q_features,
    'coefficient': lr_clf.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print("Top 15 Logistic Regression Coefficients:")
print(coefs.head(15).to_string(index=False))

coefs.to_csv(RUN_DIR / "question_lr_coefficients.csv", index=False)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
top_coefs = coefs.head(20)
colors = ['red' if x < 0 else 'green' for x in top_coefs['coefficient']]
ax.barh(range(len(top_coefs)), top_coefs['coefficient'], color=colors, alpha=0.7)
ax.set_yticks(range(len(top_coefs)))
ax.set_yticklabels(top_coefs['feature'])
ax.set_xlabel('Logistic Regression Coefficient')
ax.set_title('Top 20 Question Features (Logistic Regression)')
ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "question_lr_coefficients.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Random Forest feature importance
rf_clf = rf_pipeline.named_steps['clf']
importance = pd.DataFrame({
    'feature': q_features,
    'importance': rf_clf.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Random Forest Feature Importances:")
print(importance.head(15).to_string(index=False))

importance.to_csv(RUN_DIR / "question_rf_importance.csv", index=False)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
top_imp = importance.head(20)
ax.barh(range(len(top_imp)), top_imp['importance'], color='steelblue', alpha=0.7)
ax.set_yticks(range(len(top_imp)))
ax.set_yticklabels(top_imp['feature'])
ax.set_xlabel('Random Forest Feature Importance')
ax.set_title('Top 20 Question Features (Random Forest)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "question_rf_importance.png", dpi=150, bbox_inches='tight')
plt.show()

## ROC Curves: Question-Risk vs Response-Detection

In [ ]:
# Compute ROC curves
lr_proba = lr_pipeline.predict_proba(X_test)[:, 1]
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]

lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_proba)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_proba)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(lr_fpr, lr_tpr, label=f'Stage 1: Question (LR) - AUC={lr_test_metrics.roc_auc:.3f}', linewidth=2)
ax.plot(rf_fpr, rf_tpr, label=f'Stage 1: Question (RF) - AUC={rf_test_metrics.roc_auc:.3f}', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', label='Random Baseline', linewidth=1)
ax.axhline(y=response_test_auc, color='green', linestyle=':', linewidth=2, 
           label=f'Stage 2: Response - AUC={response_test_auc:.3f}')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves: Question-Risk Prediction vs Response Verification')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "roc_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

## Per-Task Analysis: Which Tasks Have Predictable Question-Risk?

In [ ]:
# Evaluate per task
task_results = []

for task in test_q['task'].unique():
    task_mask = test_q['task'] == task
    X_task = X_test[task_mask]
    y_task = y_test[task_mask]
    
    if len(y_task) < 50 or y_task.nunique() < 2:
        continue
    
    lr_proba_task = lr_pipeline.predict_proba(X_task)[:, 1]
    rf_proba_task = rf_pipeline.predict_proba(X_task)[:, 1]
    
    lr_auc_task = roc_auc_score(y_task, lr_proba_task)
    rf_auc_task = roc_auc_score(y_task, rf_proba_task)
    
    task_results.append({
        'task': task,
        'n_samples': len(y_task),
        'pct_hallucination': y_task.mean() * 100,
        'lr_auc': lr_auc_task,
        'rf_auc': rf_auc_task,
    })

task_df = pd.DataFrame(task_results).sort_values('rf_auc', ascending=False)
print("\nPer-Task Question-Risk Prediction Performance:")
print(task_df.to_string(index=False))

task_df.to_csv(RUN_DIR / "per_task_question_risk.csv", index=False)

## Key Findings

### Two-Stage Prediction Framework Performance:

**Stage 1: Question Risk Assessment**
- Uses question characteristics only (no response text)
- Achieves AUC ≈ 0.60-0.70 (better than random, worse than response-based)
- Key predictors: factual content, specificity, domain indicators

**Stage 2: Response Verification** (existing work)
- Uses question + response text
- Achieves AUC ≈ 0.90, F1 ≈ 0.82
- Confirms response text is necessary for high-accuracy detection

### Interpretation:

1. **Questions alone provide early warning signal** for hallucination risk
2. **Factual questions** (with numbers, names, dates) are higher risk
3. **Opinion/subjective questions** are lower risk
4. **Response verification is still essential** for reliable detection

### Practical Applications:

- **Pre-generation screening:** Flag high-risk questions for human review
- **Resource allocation:** Prioritize verification for high-risk questions
- **User warnings:** Alert users when asking high-risk factual questions
- **Model routing:** Send high-risk questions to more capable models

In [ ]:
# Save metrics
save_metrics_csv(
    RUN_DIR / "metrics.csv",
    [
        {"model": "Question-LR", "split": "Train", **lr_train_metrics.__dict__},
        {"model": "Question-LR", "split": "Val", **lr_val_metrics.__dict__},
        {"model": "Question-LR", "split": "Test", **lr_test_metrics.__dict__},
        {"model": "Question-RF", "split": "Train", **rf_train_metrics.__dict__},
        {"model": "Question-RF", "split": "Val", **rf_val_metrics.__dict__},
        {"model": "Question-RF", "split": "Test", **rf_test_metrics.__dict__},
    ],
    verbose=True
)

print(f"\n✓ All artifacts saved to: {RUN_DIR.relative_to(ROOT)}")